In [1]:
!rm -r /kaggle/working/*

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/lsa-x-video/data/info/videolibros_private.txt
/kaggle/input/lsa-x-video/data/subtitles/kaggle/working/output/0-163.vtt
/kaggle/input/lsa-x-video/data/subtitles/kaggle/working/output/1-61.vtt
/kaggle/input/lsa-x-video/data/subtitles/kaggle/working/output/0-46.vtt
/kaggle/input/lsa-x-video/data/subtitles/kaggle/working/output/0-57.vtt
/kaggle/input/lsa-x-video/data/subtitles/kaggle/working/output/0-168.vtt
/kaggle/input/lsa-x-video/data/subtitles/kaggle/working/output/2-40.vtt
/kaggle/input/lsa-x-video/data/subtitles/kaggle/working/output/1-42.vtt
/kaggle/input/lsa-x-video/data/subtitles/kaggle/working/output/0-288.vtt
/kaggle/input/lsa-x-video/data/subtitles/kaggle/working/output/0-26.vtt
/kaggle/input/lsa-x-video/data/subtitles/kaggle/working/output/0-50.vtt
/kaggle/input/lsa-x-video/data/subtitles/kaggle/working/output/2-8.vtt
/kaggle/input/lsa-x-video/data/subtitles/kaggle/working/output/0-146.vtt
/kaggle/input/lsa-x-video/data/subtitles/kaggle/working/output/0-330.vtt


In [3]:
!wget https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11m.pt -O /kaggle/working/yolo11m.pt

--2025-12-12 19:02:11--  https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11m.pt
Resolving github.com (github.com)... 140.82.116.3
Connecting to github.com (github.com)|140.82.116.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/521807533/dcd82112-f746-4e5c-a1d7-f696e642933e?sp=r&sv=2018-11-09&sr=b&spr=https&se=2025-12-12T19%3A50%3A19Z&rscd=attachment%3B+filename%3Dyolo11m.pt&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2025-12-12T18%3A49%3A58Z&ske=2025-12-12T19%3A50%3A19Z&sks=b&skv=2018-11-09&sig=wkqzVxXdjf0rdUaMdrv9bAJWMLYo3QzViOd8N5l%2Bx%2Bw%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc2NTU2NzkzMSwibmJmIjoxNzY1NTY2MTMxLCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZHVjdGlvbi5ibG9iLmNvc

In [4]:
!pip install webvtt-py ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.3 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of mkl-fft to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of mkl-random to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of mkl-umath to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 26.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 115.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 98.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 81.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
import os
from ultralytics import YOLO
from pathlib import Path
import cv2
import json
import queue
import threading
from tqdm.notebook import tqdm
from video import read_cap_segments

CLIP_PADDING = 1.7
CLIP_OVERLAP_GAP = 0.3
BATCH_SIZE = 32

def process_folder(PATH):
    os.makedirs(f"/kaggle/working/{PATH.name}/unlabeled", exist_ok=True)
    
    def process_video(f, OUTPUT_PATH, unlabeled):
        # Create a YOLO model
        model = YOLO("/kaggle/working/yolo11m.pt")
        batch_queue = queue.Queue(maxsize = 32)
        
        def producer():
            current_batch_frames = []
            current_batch_ts = []
            cap = cv2.VideoCapture(PATH / ("unlabeled" if unlabeled else "video") / f)
            for frame, timestamp in read_cap_segments(cap):
                current_batch_frames.append(frame)
                current_batch_ts.append(timestamp)
                if len(current_batch_frames) == BATCH_SIZE:
                    batch_queue.put((current_batch_frames, current_batch_ts))
                    current_batch_frames = []
                    current_batch_ts = []
            if len(current_batch_frames) != 0:
                batch_queue.put((current_batch_frames, current_batch_ts))

        def consumer():
            video_output = []
            while True:
                item = batch_queue.get()
                if item is None:
                    break
                results = model.track(item[0], persist=True, classes=[0], verbose=False, batch=len(item[0]))
                for idx, r in enumerate(results):
                    if r.boxes is not None and r.boxes.id is not None:
                        ids = r.boxes.id.cpu().tolist()
                        xyxy = r.boxes.xyxy.cpu().tolist()
                        boxes_dict = {}
                        for track_id, box in zip(ids, xyxy):
                            boxes_dict[track_id] = box
                        video_output.append({
                            "timestamp": item[1][idx],
                            "boxes": boxes_dict
                        })
            json.dump(video_output, open(OUTPUT_PATH / f"{f.replace('.mp4', '.json')}", "w"))

        t1 = threading.Thread(target=producer)
        t2 = threading.Thread(target=consumer)
        t1.start()
        t2.start()
        t1.join()
        batch_queue.put(None)
        t2.join()
    
    files = os.listdir(PATH / "video")
    for f in tqdm(files):
        process_video(f, Path(f"/kaggle/working/{PATH.name}"), False)

    unlabeled = os.listdir(PATH / "unlabeled")
    for f in tqdm(unlabeled):
        process_video(f, Path(f"/kaggle/working/{PATH.name}/unlabeled"), True)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
%%time
folders = ["0-AsociacionCivil", "1-videolibros_private", "2-videolibros_public", "3-CNSordos", "4-Locufre"]
process_folder(Path("/kaggle/input/lsa-x-video/data/raw/" + folders[3]))

  0%|          | 0/67 [00:00<?, ?it/s]

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.11.13 environment at: /usr
Resolved 2 packages in 173ms
Prepared 1 package in 50ms
Installed 1 package in 4ms
 + lap==0.5.12

requirements: AutoUpdate success ✅ 0.6s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

WARNING ⚠️ not enough matching points
WARNING ⚠️ not enough matching points
WARNING ⚠️ not enough matching points
WARNING ⚠️ not enough matching points
WARNING ⚠️ not enough matching points
WARNING ⚠️ not enough matching points
WARNING ⚠️ not enough matching points
WARNING ⚠️ not enough matching points
WARNING ⚠️ not enough matching points
WARNING ⚠️ not enough matching points
WARNING ⚠️ not enough matching points
WARNING ⚠️ not enough matching points
WARNING ⚠️ not enough matching points
WARNING ⚠️ not enough matching points
WARNING ⚠️ not enough matching points
WARNING ⚠️ not enough matching points
WARNING ⚠️ not enough matching poin